# WR Top-10 Features - All Final Models

Svi finalni modeli (RF, XGB, LGB, ElasticNet, MLP Hybrid, MLP Quantile, LSTM Improved, GRU Improved)
trenirani koristeci samo top-10 obelezja iz `results/selected_features_top40.json`
(rangirano po `combined_score` iz XGB+LGB importance).

## Odabranih 10 obelezja

1. `first_downs_roll5`
2. `target_share_std_lag1`
3. `targets_roll5`
4. `air_yards_roll5`
5. `pregame_total`
6. `avg_yards_last_season`
7. `avg_start_yardline_roll5`
8. `wind_mph`
9. `target_share_std_roll5`
10. `air_yard_share_roll5`

## Modeli

Tabular (top-10, sqrt target, sample weights=0.6):
1. RandomForest
2. XGBoost
3. LightGBM
4. ElasticNet
5. MLP Hybrid (Huber + GaussianNoise)
6. MLP Quantile q50 (pinball loss)

Sekvencni (seq_len=12, top-10 kao channels, sqrt target):
7. LSTM Improved - best HPs iz `WR_RNN_Improved` (Optuna best, test MAE 18.03)
8. GRU Improved - best HPs iz `WR_RNN_Improved_GRU` (Optuna best, test MAE 18.09)

## Cilj

Videti koliko pada (ili raste) performans kad drasticno smanjimo broj obelezja sa 40 na 10.
Nema Optune, nema tuninga - cista finalna arhitektura.


---
## 1. Imports


In [1]:
import warnings
warnings.filterwarnings('ignore')

import os, json, time, random
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, optimizers, losses, Model, Input
from tensorflow.keras.layers import (
    Dense, Dropout, LayerNormalization, GaussianNoise,
    LSTM, GRU, Bidirectional, Masking, Concatenate,
)

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100
matplotlib.rcParams['figure.figsize'] = (14, 5)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'

for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

print(f'TensorFlow {tf.__version__}')


TensorFlow 2.21.0


---
## 2. Feature Engineering

Puni pipeline iz `WR_Ensemble_Final` (lag1/roll5 + momentum + interakcije) da bi
sve top-40 feature-a bile dostupne. Iz njih cemo onda izabrati samo top-10.


In [2]:
df = pd.read_csv('../data/fully combined/wr_all_weeks.csv')
df['week'] = df['game_id'].str.split('_').str[1].astype(int)
df = df.sort_values(['receiver_player_id', 'season', 'week']).reset_index(drop=True)

work_df = df.copy()

wp_rename_map = {
    'yards_wp_<25':       'yards_wp_less_than_25',
    'yards_wp_>75':       'yards_wp_greater_than_75',
    'receptions_wp_<25':  'receptions_wp_less_than_25',
    'receptions_wp_>75':  'receptions_wp_greater_than_75',
    'targets_wp_<25':     'targets_wp_less_than_25',
    'targets_wp_>75':     'targets_wp_greater_than_75',
}
work_df = work_df.rename(columns={k: v for k, v in wp_rename_map.items() if k in work_df.columns})

work_df['player_team_inferred'] = np.where(
    (work_df['home_team'] == work_df['defteam']) & (work_df['away_team'] != work_df['defteam']),
    work_df['away_team'],
    np.where(
        (work_df['away_team'] == work_df['defteam']) & (work_df['home_team'] != work_df['defteam']),
        work_df['home_team'], np.nan,
    ),
)
prev_team = work_df.groupby('receiver_player_id')['player_team_inferred'].shift(1)
work_df['team_changed'] = (
    work_df['player_team_inferred'].notna()
    & prev_team.notna()
    & (work_df['player_team_inferred'] != prev_team)
).astype(int)

prev_season = work_df.groupby('receiver_player_id')['season'].shift(1)
work_df['is_new_season'] = (prev_season.notna() & (work_df['season'] != prev_season)).astype(int)

work_df['season_week_abs'] = (work_df['season'] - 2015) * 22 + work_df['week']
work_df['weeks_since_last_game'] = (
    work_df.groupby('receiver_player_id')['season_week_abs']
    .diff().fillna(1).clip(lower=1).astype(float)
)
work_df.drop(columns=['season_week_abs'], inplace=True)

season_career = (
    work_df.groupby(['receiver_player_id', 'season'], as_index=False)
    .agg(avg_yards=('receiving_yards', 'mean'),
         avg_target_share=('target_share', 'mean'),
         avg_epa=('epa', 'mean'),
         avg_air_yard_share=('air_yard_share', 'mean'),
         avg_catch_rate=('catch_rate', 'mean'),
         games_played=('game_id', 'count'))
    .sort_values(['receiver_player_id', 'season'])
)
for c in ['avg_yards', 'avg_target_share', 'avg_epa',
          'avg_air_yard_share', 'avg_catch_rate', 'games_played']:
    season_career[f'{c}_last_season'] = (
        season_career.groupby('receiver_player_id')[c].shift(1)
    )
career_cols = [f'{c}_last_season' for c in
               ['avg_yards', 'avg_target_share', 'avg_epa',
                'avg_air_yard_share', 'avg_catch_rate', 'games_played']]
season_career = season_career[['receiver_player_id', 'season'] + career_cols]
work_df = work_df.merge(season_career, on=['receiver_player_id', 'season'], how='left')
work_df[career_cols] = work_df[career_cols].fillna(0)

rolling_source_cols = [
    'targets', 'receptions', 'air_yards', 'yac', 'tds', 'epa', 'wpa', 'catch_rate',
    'avg_depth', 'adot', 'yac_per_reception', 'td_rate', 'explosive_plays', 'first_downs',
    'yards_per_target', 'team_pass_attempts', 'team_air_yards', 'team_epa', 'air_yard_share',
    'target_share', 'qb_completions', 'qb_attempts', 'qb_air_yards', 'qb_cpoe', 'qb_comp_pct',
    'avg_score_diff', 'trailing_pct', 'leading_pct', 'avg_quarter', 'success_rate',
    'big_play_rate', 'avg_start_yardline', 'red_zone_targets', 'end_zone_targets',
    'third_down_targets', 'fourth_down_targets', 'high_leverage_targets',
    'second_and_long_targets', 'third_and_medium_targets', 'wp_var', 'target_share_std',
    'reception_std', 'def_targets_dev', 'def_receptions_dev', 'def_yards_dev', 'def_tds_dev',
    'def_epa_dev', 'yards_Q1', 'yards_Q2', 'yards_Q3', 'yards_Q4',
    'receptions_Q1', 'receptions_Q2', 'receptions_Q3', 'receptions_Q4',
    'targets_Q1', 'targets_Q2', 'targets_Q3', 'targets_Q4',
    'lost_yards_due_to_penalty',
    'yards_wp_less_than_25', 'yards_wp_25_45', 'yards_wp_45_55',
    'yards_wp_55_75', 'yards_wp_greater_than_75',
    'receptions_wp_less_than_25', 'receptions_wp_25_45', 'receptions_wp_45_55',
    'receptions_wp_55_75', 'receptions_wp_greater_than_75',
    'targets_wp_less_than_25', 'targets_wp_25_45', 'targets_wp_45_55',
    'targets_wp_55_75', 'targets_wp_greater_than_75',
    'weeks_since_last_game',
]
available_roll_cols = [c for c in rolling_source_cols if c in work_df.columns]
grp = work_df.groupby('receiver_player_id', sort=False)
derived_cols = []
for col in available_roll_cols:
    work_df[f'{col}_lag1']  = grp[col].transform(lambda s: s.shift(1))
    work_df[f'{col}_roll5'] = grp[col].transform(
        lambda s: s.shift(1).rolling(window=5, min_periods=1).mean()
    )
    derived_cols.extend([f'{col}_lag1', f'{col}_roll5'])

momentum_sources = [
    'targets', 'receptions', 'air_yards', 'epa', 'catch_rate',
    'target_share', 'yards_per_target', 'air_yard_share',
]
momentum_cols = []
for col in momentum_sources:
    lag1_col, roll5_col = f'{col}_lag1', f'{col}_roll5'
    if lag1_col in work_df.columns and roll5_col in work_df.columns:
        mcol = f'{col}_momentum'
        work_df[mcol] = work_df[lag1_col] - work_df[roll5_col]
        momentum_cols.append(mcol)

interaction_cols = []
if 'target_share_lag1' in work_df.columns:
    work_df['target_volume_interaction'] = (
        work_df['target_share_lag1'] * work_df['pregame_total']
    )
    interaction_cols.append('target_volume_interaction')

pregame_features = [
    'pregame_spread', 'pregame_total', 'surface', 'is_dome', 'temp_f',
    'humidity_pct', 'wind_mph', 'is_rain', 'is_snow', 'is_clear',
    'season', 'week', 'team_changed', 'is_new_season',
]
all_feature_columns = (
    pregame_features + career_cols + derived_cols + momentum_cols + interaction_cols
)

model_df = work_df.copy()
lag1_all = [c for c in all_feature_columns if c.endswith('_lag1')]
model_df = model_df.loc[~model_df[lag1_all].isna().all(axis=1)].copy()
model_df = model_df.loc[:, ~model_df.columns.duplicated()]
model_df[all_feature_columns] = model_df[all_feature_columns].fillna(0)
model_df['receiving_yards_sqrt'] = np.sqrt(model_df['receiving_yards'].clip(lower=0))

print(f'Dataset: {model_df.shape}')
print(f'Total features engineered: {len(all_feature_columns)}')


Dataset: (44396, 272)
Total features engineered: 181


---
## 3. Top-10 Feature Selection + Split + Scaler


In [3]:
selected_features = [
    'first_downs_roll5',
    'target_share_std_lag1',
    'targets_roll5',
    'air_yards_roll5',
    'pregame_total',
    'avg_yards_last_season',
    'avg_start_yardline_roll5',
    'wind_mph',
    'target_share_std_roll5',
    'air_yard_share_roll5',
]

print(f'Using top-10 features:')
for i, f in enumerate(selected_features):
    print(f'  {i+1}. {f}')

missing = [f for f in selected_features if f not in model_df.columns]
if missing:
    raise ValueError(f'Missing features in engineered dataset: {missing}')

train_seasons = list(range(2015, 2022))
val_seasons   = [2022, 2023]
test_seasons  = [2024, 2025]

train_df = model_df[model_df['season'].isin(train_seasons)].copy()
val_df   = model_df[model_df['season'].isin(val_seasons)].copy()
test_df  = model_df[model_df['season'].isin(test_seasons)].copy()

y_train_sqrt = train_df['receiving_yards_sqrt'].values
y_val_sqrt   = val_df['receiving_yards_sqrt'].values
y_test_sqrt  = test_df['receiving_yards_sqrt'].values
y_train_orig = train_df['receiving_yards'].values
y_val_orig   = val_df['receiving_yards'].values
y_test_orig  = test_df['receiving_yards'].values

scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[selected_features].values)
X_val   = scaler.transform(val_df[selected_features].values)
X_test  = scaler.transform(test_df[selected_features].values)

mean_y_train = np.mean(np.clip(y_train_orig, 0, None))
sw_train = 1.0 + 0.6 * np.sqrt(np.clip(y_train_orig, 0, None) / mean_y_train)

print()
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')
print(f'Features: {X_train.shape[1]} (top-10)')
print(f'Sample weights range: [{sw_train.min():.2f}, {sw_train.max():.2f}]')


Using top-10 features:
  1. first_downs_roll5
  2. target_share_std_lag1
  3. targets_roll5
  4. air_yards_roll5
  5. pregame_total
  6. avg_yards_last_season
  7. avg_start_yardline_roll5
  8. wind_mph
  9. target_share_std_roll5
  10. air_yard_share_roll5

Train: (29276, 10)  Val: (8921, 10)  Test: (6199, 10)
Features: 10 (top-10)
Sample weights range: [1.00, 2.87]


---
## 4. Utility Functions


In [4]:
def eval_preds(y_true_orig, y_pred_orig):
    return {
        'MAE':  float(mean_absolute_error(y_true_orig, y_pred_orig)),
        'RMSE': float(np.sqrt(mean_squared_error(y_true_orig, y_pred_orig))),
        'R2':   float(r2_score(y_true_orig, y_pred_orig)),
    }

def sqrt_to_orig(p):
    return np.clip(p, 0, None) ** 2

results = {}

def log_result(name, val_metrics, test_metrics, seconds=None):
    results[name] = {'val': val_metrics, 'test': test_metrics, 'seconds': seconds}
    s = f' ({seconds:.0f}s)' if seconds is not None else ''
    print(f'{name:32}{s}  val MAE={val_metrics["MAE"]:.3f}  '
          f'test MAE={test_metrics["MAE"]:.3f}  R2={test_metrics["R2"]:.4f}')

print('Utils defined.')


Utils defined.


---
## 5. Tabular Models


### 5.1 RandomForest


In [5]:
t0 = time.time()
rf = RandomForestRegressor(
    n_estimators=300, max_depth=10, min_samples_leaf=4,
    random_state=SEED, n_jobs=-1,
)
rf.fit(X_train, y_train_sqrt, sample_weight=sw_train)
rf_val_pred  = sqrt_to_orig(rf.predict(X_val))
rf_test_pred = sqrt_to_orig(rf.predict(X_test))
log_result('RandomForest',
           eval_preds(y_val_orig,  rf_val_pred),
           eval_preds(y_test_orig, rf_test_pred),
           time.time() - t0)


RandomForest                     (8s)  val MAE=18.224  test MAE=17.898  R2=0.3247


### 5.2 XGBoost


In [6]:
t0 = time.time()
xgb = XGBRegressor(
    n_estimators=800, max_depth=6, learning_rate=0.03,
    subsample=0.85, colsample_bytree=0.85,
    min_child_weight=4, gamma=0.1,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=SEED, tree_method='hist', verbosity=0,
)
xgb.fit(X_train, y_train_sqrt, sample_weight=sw_train,
        eval_set=[(X_val, y_val_sqrt)], verbose=False)
xgb_val_pred  = sqrt_to_orig(xgb.predict(X_val))
xgb_test_pred = sqrt_to_orig(xgb.predict(X_test))
log_result('XGBoost',
           eval_preds(y_val_orig,  xgb_val_pred),
           eval_preds(y_test_orig, xgb_test_pred),
           time.time() - t0)


XGBoost                          (4s)  val MAE=18.315  test MAE=18.049  R2=0.3111


### 5.3 LightGBM


In [7]:
t0 = time.time()
lgb = LGBMRegressor(
    n_estimators=1000, num_leaves=63, learning_rate=0.03,
    feature_fraction=0.85, bagging_fraction=0.85, bagging_freq=5,
    min_child_samples=20, reg_alpha=0.1, reg_lambda=0.1,
    random_state=SEED, verbosity=-1,
)
lgb.fit(X_train, y_train_sqrt, sample_weight=sw_train,
        eval_set=[(X_val, y_val_sqrt)])
lgb_val_pred  = sqrt_to_orig(lgb.predict(X_val))
lgb_test_pred = sqrt_to_orig(lgb.predict(X_test))
log_result('LightGBM',
           eval_preds(y_val_orig,  lgb_val_pred),
           eval_preds(y_test_orig, lgb_test_pred),
           time.time() - t0)


LightGBM                         (7s)  val MAE=18.525  test MAE=18.202  R2=0.2940


### 5.4 ElasticNet


In [8]:
t0 = time.time()
enet = ElasticNet(alpha=0.01, l1_ratio=0.3, max_iter=20000, random_state=SEED)
enet.fit(X_train, y_train_sqrt, sample_weight=sw_train)
enet_val_pred  = sqrt_to_orig(enet.predict(X_val))
enet_test_pred = sqrt_to_orig(enet.predict(X_test))
log_result('ElasticNet',
           eval_preds(y_val_orig,  enet_val_pred),
           eval_preds(y_test_orig, enet_test_pred),
           time.time() - t0)


ElasticNet                       (0s)  val MAE=18.531  test MAE=18.190  R2=0.3101


### 5.5 MLP Hybrid (Huber)

Arhitektura MODEL_B iz `WR_MLP_Hybrid`: `[448, 128, 320, 448, 256]` + GaussianNoise(0.15) + Huber(0.5).


In [9]:
MODEL_B_PARAMS = {
    'n_layers': 5,
    'units': [448, 128, 320, 448, 256],
    'dropout': 0.5,
    'huber_delta': 0.5,
    'lr': 0.003701177981657943,
    'weight_decay': 1.5007044511603625e-05,
    'batch_size': 32,
}

def build_hybrid_mlp(input_dim, params, loss_fn, noise_stddev=0.15):
    model = keras.Sequential()
    model.add(Input(shape=(input_dim,)))
    model.add(GaussianNoise(noise_stddev))
    for i in range(params['n_layers']):
        model.add(Dense(params['units'][i], activation='relu'))
        model.add(LayerNormalization())
        dr = params['dropout'] * (0.5 if i >= params['n_layers'] - 1 else 1.0)
        model.add(Dropout(dr))
    model.add(Dense(1))
    opt = optimizers.AdamW(learning_rate=params['lr'],
                           weight_decay=params['weight_decay'])
    model.compile(optimizer=opt, loss=loss_fn, metrics=['mae'])
    return model


t0 = time.time()
tf.keras.backend.clear_session()
tf.random.set_seed(SEED); np.random.seed(SEED); random.seed(SEED)

mlp_h = build_hybrid_mlp(X_train.shape[1], MODEL_B_PARAMS,
                         losses.Huber(delta=MODEL_B_PARAMS['huber_delta']))

cb_h = [
    callbacks.EarlyStopping(monitor='val_loss', patience=30,
                            restore_best_weights=True, min_delta=1e-4),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3,
                                patience=10, min_lr=1e-6),
]

mlp_h.fit(X_train, y_train_sqrt, sample_weight=sw_train,
          validation_data=(X_val, y_val_sqrt),
          epochs=500, batch_size=MODEL_B_PARAMS['batch_size'],
          callbacks=cb_h, verbose=0)

mlp_h_val_pred  = sqrt_to_orig(mlp_h.predict(X_val,  verbose=0).flatten())
mlp_h_test_pred = sqrt_to_orig(mlp_h.predict(X_test, verbose=0).flatten())
log_result('MLP Hybrid (Huber)',
           eval_preds(y_val_orig,  mlp_h_val_pred),
           eval_preds(y_test_orig, mlp_h_test_pred),
           time.time() - t0)



MLP Hybrid (Huber)               (436s)  val MAE=18.172  test MAE=17.918  R2=0.3234


### 5.6 MLP Quantile q50 (pinball loss)

Ista arhitektura kao Hybrid, ali sa pinball loss-om za medijan.


In [10]:
def pinball_loss_q50(y_true, y_pred):
    e = y_true - y_pred
    return tf.reduce_mean(tf.maximum(0.5 * e, (0.5 - 1.0) * e))


t0 = time.time()
tf.keras.backend.clear_session()
tf.random.set_seed(SEED); np.random.seed(SEED); random.seed(SEED)

mlp_q = build_hybrid_mlp(X_train.shape[1], MODEL_B_PARAMS, pinball_loss_q50)

cb_q = [
    callbacks.EarlyStopping(monitor='val_loss', patience=30,
                            restore_best_weights=True, min_delta=1e-4),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3,
                                patience=10, min_lr=1e-6),
]

mlp_q.fit(X_train, y_train_sqrt, sample_weight=sw_train,
          validation_data=(X_val, y_val_sqrt),
          epochs=500, batch_size=MODEL_B_PARAMS['batch_size'],
          callbacks=cb_q, verbose=0)

mlp_q_val_pred  = sqrt_to_orig(mlp_q.predict(X_val,  verbose=0).flatten())
mlp_q_test_pred = sqrt_to_orig(mlp_q.predict(X_test, verbose=0).flatten())
log_result('MLP Quantile q50',
           eval_preds(y_val_orig,  mlp_q_val_pred),
           eval_preds(y_test_orig, mlp_q_test_pred),
           time.time() - t0)


MLP Quantile q50                 (560s)  val MAE=17.934  test MAE=17.775  R2=0.2954


---
## 6. Sequence Data Prep (top-10 as channels)

Gradimo padded career sekvence gde svaki timestep sadrzi vrednost top-10 obelezja
u prethodnoj utakmici. `seq_len=12` (iz best HPs za LSTM/GRU Improved).

Napomena: ovo je kompromisna adaptacija. Original `WR_RNN_Improved` je koristio ~45
raw sekvencnih kanala. Ovde koristimo samo top-10. Ocekuje se da LSTM/GRU budu
slabiji jer je rolling/lag priroda top-obelezja vec agregirana.


In [11]:
SEQ_LEN = 12

# Scale top-N features (new scaler for sequence pipeline, fit on train only)
seq_scaler = StandardScaler().fit(
    model_df.loc[model_df['season'].isin(train_seasons), selected_features]
)
df_seq = model_df.copy()
df_seq[selected_features] = seq_scaler.transform(df_seq[selected_features])
df_seq['receiving_yards_orig'] = model_df['receiving_yards'].values


def build_padded_career_sequences(df_scaled, feat_cols, seq_len):
    X_seq, X_static, y_sqrt, y_orig, seasons_arr = [], [], [], [], []
    for pid, group in df_scaled.groupby('receiver_player_id'):
        group = group.sort_values(['season', 'week'])
        n = len(group)
        if n < 2:
            continue
        seq_vals    = group[feat_cols].values
        yards       = group['receiving_yards_orig'].values
        season_vals = group['season'].values
        for t in range(1, n):
            start = max(0, t - seq_len)
            past = seq_vals[start:t]
            if past.shape[0] < seq_len:
                pad = np.zeros((seq_len - past.shape[0], len(feat_cols)))
                past = np.vstack([pad, past])
            X_seq.append(past)
            X_static.append(seq_vals[t])
            y_sqrt.append(np.sqrt(max(yards[t], 0)))
            y_orig.append(yards[t])
            seasons_arr.append(season_vals[t])
    return (
        np.array(X_seq,    dtype=np.float32),
        np.array(X_static, dtype=np.float32),
        np.array(y_sqrt,   dtype=np.float32),
        np.array(y_orig,   dtype=np.float32),
        np.array(seasons_arr),
    )

t0 = time.time()
X_seq_all, X_static_all, y_sqrt_all, y_orig_all, seasons_all = \
    build_padded_career_sequences(df_seq, selected_features, SEQ_LEN)
print(f'Built {len(X_seq_all)} sequences in {time.time()-t0:.1f}s')
print(f'  X_seq={X_seq_all.shape}  X_static={X_static_all.shape}')

tr = np.isin(seasons_all, train_seasons)
va = np.isin(seasons_all, val_seasons)
te = np.isin(seasons_all, test_seasons)

X_seq_tr,    X_seq_va,    X_seq_te    = X_seq_all[tr],    X_seq_all[va],    X_seq_all[te]
X_static_tr, X_static_va, X_static_te = X_static_all[tr], X_static_all[va], X_static_all[te]
y_tr_sqrt,   y_va_sqrt,   y_te_sqrt   = y_sqrt_all[tr],   y_sqrt_all[va],   y_sqrt_all[te]
y_tr_orig,   y_va_orig,   y_te_orig   = y_orig_all[tr],   y_orig_all[va],   y_orig_all[te]

print(f'seq train={len(y_tr_sqrt)}  val={len(y_va_sqrt)}  test={len(y_te_sqrt)}')

N_SEQ_FEAT = X_seq_tr.shape[2]
N_STATIC_FEAT = X_static_tr.shape[1]


def make_weights(strength, y_orig):
    mean_y = np.mean(np.clip(y_orig, 0, None))
    base = np.sqrt(np.clip(y_orig, 0, None) / mean_y)
    return 1.0 + strength * base


Built 42930 sequences in 2.4s
  X_seq=(42930, 12, 10)  X_static=(42930, 10)
seq train=28156  val=8735  test=6039


---
## 7. RNN Model Builder

Isti fiksni-HP builder kao u `WR_RNN_Improved_GRU` (`build_gru_fixed`), generalizovan
da prihvati i LSTM i GRU. Bez Optune - samo cita hiperparametre iz `params` dicta.


In [12]:
def build_rnn_fixed(rnn_type, params, seq_len, n_seq_feat, n_static_feat):
    RNNCell = LSTM if rnn_type == 'LSTM' else GRU

    seq_input = Input(shape=(seq_len, n_seq_feat), name='seq_input')
    x = Masking(mask_value=0.0)(seq_input)

    for i in range(params['n_rnn_layers']):
        units = params[f'rnn_units_{i}']
        return_seq = (i < params['n_rnn_layers'] - 1)
        rnn_layer = RNNCell(
            units, return_sequences=return_seq,
            dropout=params['rnn_dropout'],
            recurrent_dropout=params['rnn_dropout'],
        )
        if params['bidirectional']:
            x = Bidirectional(rnn_layer)(x)
        else:
            x = rnn_layer(x)
        x = LayerNormalization()(x)
        x = Dropout(params['dropout'])(x)

    static_input = Input(shape=(n_static_feat,), name='static_input')
    s = GaussianNoise(params['noise_stddev'])(static_input)
    s = Dense(params['static_units'], activation='relu')(s)
    s = LayerNormalization()(s)
    s = Dropout(params['dropout'])(s)

    merged = Concatenate()([x, s])
    merged = Dense(params['dense_units'], activation='relu')(merged)
    merged = LayerNormalization()(merged)
    merged = Dropout(params['dropout'] * 0.5)(merged)
    output = Dense(1)(merged)

    model = Model(inputs=[seq_input, static_input], outputs=output)
    opt = optimizers.AdamW(learning_rate=params['lr'],
                           weight_decay=params['weight_decay'])
    model.compile(optimizer=opt,
                  loss=losses.Huber(delta=params['huber_delta']),
                  metrics=['mae'])
    return model


def train_rnn(rnn_type, params, name):
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED); np.random.seed(SEED); random.seed(SEED)

    model = build_rnn_fixed(rnn_type, params, SEQ_LEN, N_SEQ_FEAT, N_STATIC_FEAT)
    sw = make_weights(params['sw_strength'], y_tr_orig)

    cbs = [
        callbacks.EarlyStopping(monitor='val_loss', patience=30,
                                restore_best_weights=True, min_delta=1e-4),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3,
                                    patience=10, min_lr=1e-6),
    ]

    t0 = time.time()
    model.fit(
        [X_seq_tr, X_static_tr], y_tr_sqrt,
        sample_weight=sw,
        validation_data=([X_seq_va, X_static_va], y_va_sqrt),
        epochs=500, batch_size=params['batch_size'],
        callbacks=cbs, verbose=0,
    )
    elapsed = time.time() - t0

    pred_va = sqrt_to_orig(
        model.predict([X_seq_va, X_static_va], verbose=0).flatten()
    )
    pred_te = sqrt_to_orig(
        model.predict([X_seq_te, X_static_te], verbose=0).flatten()
    )
    log_result(name,
               eval_preds(y_va_orig, pred_va),
               eval_preds(y_te_orig, pred_te),
               elapsed)
    return model


### 7.1 LSTM Improved (best HPs)

Fiksirani HPs iz `WR_RNN_Improved` Optuna best trial-a:
`seq_len=12, 1 LSTM layer x 128 units, unidirectional, dropout=0.3, Huber(0.5), lr=3.5e-4`.


In [13]:
LSTM_PARAMS = {
    "seq_len": 12,
    "batch_size": 32,
    "sw_strength": 0.6,
    "bidirectional": False,
    "n_rnn_layers": 1,
    "dropout": 0.3,
    "rnn_dropout": 0.3,
    "noise_stddev": 0.2,
    "huber_delta": 0.5,
    "lr": 0.00034978,
    "weight_decay": 0.0019053,
    "rnn_units_0": 128,
    "static_units": 96,
    "dense_units": 64
}

_ = train_rnn('LSTM', LSTM_PARAMS, 'LSTM Improved')


LSTM Improved                    (972s)  val MAE=18.288  test MAE=18.192  R2=0.3206


### 7.2 GRU Improved (best HPs)

Fiksirani HPs iz `WR_RNN_Improved_GRU` Optuna best trial-a:
`seq_len=12, 1 GRU layer x 192 units, unidirectional, dropout=0.35, Huber(2.0), lr=1.4e-4`.


In [14]:
GRU_PARAMS = {
    "seq_len": 12,
    "batch_size": 64,
    "sw_strength": 0.7,
    "bidirectional": False,
    "n_rnn_layers": 1,
    "dropout": 0.35,
    "rnn_dropout": 0.1,
    "noise_stddev": 0.2,
    "huber_delta": 2.0,
    "lr": 0.00013876,
    "weight_decay": 0.000498,
    "rnn_units_0": 192,
    "static_units": 32,
    "dense_units": 64
}

_ = train_rnn('GRU', GRU_PARAMS, 'GRU Improved')


GRU Improved                     (524s)  val MAE=18.475  test MAE=18.329  R2=0.3192


---
## 8. Final Comparison Table - Top-10


In [15]:
rows = []
for name, r in results.items():
    rows.append({
        'Model':     name,
        'Val MAE':   round(r['val']['MAE'],   3),
        'Test MAE':  round(r['test']['MAE'],  3),
        'Test RMSE': round(r['test']['RMSE'], 3),
        'Test R2':   round(r['test']['R2'],   4),
        'Seconds':   round(r['seconds'], 1) if r.get('seconds') else None,
    })
summary = pd.DataFrame(rows).sort_values('Test MAE').reset_index(drop=True)
print(f'=== WR Top-10 Features — Final Comparison ===')
print(summary.to_string(index=False))

os.makedirs('../results', exist_ok=True)
summary.to_csv('../results/wr_top10_models_summary.csv', index=False)
print()
print(f'Saved: ../results/wr_top10_models_summary.csv')


=== WR Top-10 Features — Final Comparison ===
             Model  Val MAE  Test MAE  Test RMSE  Test R2  Seconds
  MLP Quantile q50   17.934    17.775     25.855   0.2954    560.1
      RandomForest   18.224    17.898     25.313   0.3247      8.4
MLP Hybrid (Huber)   18.172    17.918     25.336   0.3234    435.6
           XGBoost   18.315    18.049     25.565   0.3111      3.9
        ElasticNet   18.531    18.190     25.584   0.3101      0.1
     LSTM Improved   18.288    18.192     25.485   0.3206    972.3
          LightGBM   18.525    18.202     25.882   0.2940      7.2
      GRU Improved   18.475    18.329     25.512   0.3192    523.9

Saved: ../results/wr_top10_models_summary.csv
